In [ ]:
!pip install wikipedia pandas nltk sentence-transformers

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=60352fbee80e400a640b7a7e0596d3fe9cdba1e061727475602ae54fb2ba5aaf
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [ ]:
import wikipedia
import pandas as pd
import nltk
import re

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

**Base Topics**

In [ ]:
base_topics = [
    "Black hole",
    "Galaxy",
    "Planet",
    "Universe",
    "Star",
    "Gravity",
    "Space exploration"
]

**AUTO EXPAND TOPICS**

In [ ]:
expanded_topics = set()

for topic in base_topics:
    try:
        results = wikipedia.search(topic, results=10)
        for r in results:
            expanded_topics.add(r)
    except:
        pass

topics = list(expanded_topics)
print("Total topics:", len(topics))

Total topics: 70


**EXTRACT DATA**

In [ ]:
data = []

for topic in topics:
    try:
        page = wikipedia.page(topic, auto_suggest=False)
        text = page.content

        data.append({
            "topic": topic,
            "text": text
        })

    except:
        continue

/usr/local/lib/python3.12/dist-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /usr/local/lib/python3.12/dist-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


**SPLIT + FILTER**

In [ ]:
from nltk.tokenize import sent_tokenize

rows = []

for item in data:
    sentences = sent_tokenize(item['text'])

    for s in sentences:
        if 50 < len(s) < 300:
            rows.append({
                "topic": item['topic'],
                "complex_text": s
            })

**CLEAN**

In [ ]:
import re

def clean(text):
    text = re.sub(r'\[[0-9]*\]', '', text)
    text = re.sub(r'\n', ' ', text)
    return text

df = pd.DataFrame(rows)
df['complex_text'] = df['complex_text'].apply(clean)

In [ ]:
print("Total rows:", len(df))

Total rows: 13338


In [ ]:
df.to_csv("space_dataset.csv", index=False)

In [ ]:
from google.colab import files
files.download("space_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Remove Duplicates**

In [ ]:
df = df.drop_duplicates(subset='complex_text')

**Remove Short/Long Sentences**

In [ ]:
df = df[df['complex_text'].str.len() > 50]
df = df[df['complex_text'].str.len() < 300]

**Basic Text Cleaning**

In [ ]:
import re

def clean(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9., ]', '', text)
    return text

df['clean_text'] = df['complex_text'].apply(clean)

In [ ]:
print("Total rows:", len(df))

Total rows: 13262


**Tokenization + Lemmatization**

In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

nltk.download('stopwords')
nltk.download('wordnet')
# nltk.download('punkt')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    words = nltk.word_tokenize(text)
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return " ".join(words)

df['processed_text'] = df['clean_text'].apply(preprocess)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


**TF-IDF**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
tfidf_matrix = tfidf.fit_transform(df['processed_text'])

**BERT Embeddings**

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for the complex_text column
embeddings = model.encode(df['complex_text'].tolist(), show_progress_bar=True)

print("Embeddings shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/415 [00:00<?, ?it/s]

Embeddings shape: (13262, 384)


In [ ]:
import numpy as np
np.save("embeddings.npy", embeddings)
df.to_csv("dataset_cleaned.csv", index=False)

In [ ]:
print(type(embeddings))
print(embeddings[0][:10])

<class 'numpy.ndarray'>
[ 0.04073058 -0.05986582  0.07493372 -0.00650114 -0.06705786 -0.04111829
  0.03974353 -0.05542217  0.00592894  0.00578257]


In [ ]:
# from sentence_transformers import util

# def retrieve(query):
#     # convert query → embedding
#     q_emb = model.encode(query)

#     # compute similarity
#     scores = util.cos_sim(q_emb, embeddings)

#     # get best match index and convert to integer
#     idx = scores.argmax().item()

#     return df.iloc[idx]['complex_text']
from sentence_transformers import util

def retrieve_top_k(query, k=1):
    q_emb = model.encode(query)
    scores = util.cos_sim(q_emb, embeddings)[0]

    top_k = scores.argsort(descending=True)[:k].tolist()

    return " ".join(df.iloc[i]['complex_text'] for i in top_k)

In [ ]:
print(retrieve_top_k("what is a black hole"))
print(retrieve_top_k("explain gravity"))
print(retrieve_top_k("what is galaxy"))

Black holes are a class of astronomical objects that have undergone gravitational collapse, leaving behind spheroidal regions of space that nothing, not even light, can escape.
In physics, gravity (from Latin  gravitas 'weight'), also known as gravitation or a gravitational interaction, is a fundamental interaction, which may be described as the force that draws material objects towards each other.
A galaxy is a system of stars, stellar remnants, interstellar gas, dust, and dark matter bound together by gravity.


**STEP 2:Simplification using T5**

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize

def split_sentences(text):
    return sent_tokenize(text)

def simplify_sentence(sentence):

    prompt = (
        "Rewrite the following sentence in simple English. "
    "Do NOT add any new facts. Do NOT change meaning.\n\n"
        + sentence
    )

    inputs = t5_tokenizer(prompt, return_tensors="pt", truncation=True)

    outputs = t5_model.generate(
        **inputs,
        max_length=70,
        num_beams=4,
        repetition_penalty=1.3,
        no_repeat_ngram_size=2,
        early_stopping=True
    )

    return t5_tokenizer.decode(outputs[0], skip_special_tokens=True)

def simplify(text):
    sentences = split_sentences(text)
    simplified = [simplify_sentence(s) for s in sentences]
    return " ".join(simplified)

In [ ]:
text = retrieve_top_k("what is black hole")

print("Original:", text)
print("Simplified:", simplify(text))

Original: Black holes are a class of astronomical objects that have undergone gravitational collapse, leaving behind spheroidal regions of space that nothing, not even light, can escape.
Simplified: Black holes are astronomical objects that have undergone gravitational collapsibility collapse, leaving behind spiral regions of space.


In [ ]:
def detect_intent(query):
    query = query.lower()

    if "simple" in query or "easy" in query:
        return "simple"

    elif "what is" in query or "define" in query:
        return "define"

    elif "explain" in query:
        return "explain"

    else:
        return "explain"

In [ ]:
import re

def post_process(text):
    text = text.strip()

    # Fix spacing issues
    text = re.sub(r'\s+', ' ', text)

    # Capitalize first letter
    if len(text) > 0:
        text = text[0].upper() + text[1:]

    # Ensure proper sentence ending
    if len(text) > 0 and text[-1] not in ".!?":
        text += "."

    return text

In [ ]:
def chatbot(query):
    # Detect intent (optional but kept for structure)
    intent = detect_intent(query)

    # Step 1: Retrieve relevant text
    text = retrieve_top_k(query, k=1)

    # Step 2: Simplify text
    response = simplify(text)

    # Step 3: Comprehensive post-processing (Formatting + Grammar)
    final_output = post_process(response)

    return final_output

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Input box
query_input = widgets.Text(
    placeholder='Enter your question',
    description='Query:',
    layout=widgets.Layout(width='70%')
)

# Button
button = widgets.Button(
    description="Get Answer",
    button_style='success'
)

# Output area
output = widgets.Output()

def on_click(b):
    with output:
        output.clear_output()
        query = query_input.value

        if query.strip() == "":
            print("⚠️ Please enter a question.")
        else:
            print("⏳ Processing...\n")
            response = chatbot(query)
            print("🧠 Answer:\n")
            print(response)

button.on_click(on_click)

display(query_input, button, output)

Text(value='', description='Query:', layout=Layout(width='70%'), placeholder='Enter your question')

Button(button_style='success', description='Get Answer', style=ButtonStyle())

Output()

In [ ]:
print("Query: Explain black hole")
print("Response:", chatbot("Explain black hole"))
print("\n---\n")
print("Query: What is gravity")
print("Response:", chatbot("What is gravity"))
print("\n---\n")
print("Query: Explain galaxy in simple terms")
print("Response:", chatbot("Explain galaxy in simple terms"))

Query: Explain black hole
Response: Black holes are astronomical objects that have undergone gravitational collapsibility collapse, leaving behind spiral regions of space.

---

Query: What is gravity
Response: Gravitational is a fundamental interaction, which may be described as the force that draws material objects towards each other.

---

Query: Explain galaxy in simple terms
Response: A galaxie is a system of stars, stellar remnants and intersector gases.
